<a href="https://colab.research.google.com/github/Surajsurya95096/My-Bot-Deployer/blob/main/deploy.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# @title 🚀 **Heroku Deployer** { display-mode: "form" }

# @markdown ### ⚙️ Heroku Setup
Heroku_Email = ""  # @param {type:"string"}
Heroku_API_Key = ""  # @param {type:"string"}
Heroku_App_Name = ""  # @param {type:"string"}

# @markdown ---
# @markdown ### 🔒 GitHub Settings
Git_Repo_URL = ""  # @param {type:"string"}
Git_Branch = ""  # @param {type:"string"}
GitHub_Access_Token = ""  # @param {type:"string"}

# @markdown ---
# @markdown ### 🤖 Variables
BOT_TOKEN = ""  # @param {type:"string"}
API_ID = ""  # @param {type:"string"}
API_HASH = ""  # @param {type:"string"}
OWNER_ID = ""  # @param {type:"string"}
DATABASE_URL = ""  # @param {type:"string"}

import os
import subprocess
import sys
import time

os.chdir("/content")

GREEN = "\033[92m"
RED = "\033[91m"
YELLOW = "\033[93m"
RESET = "\033[0m"

def run_cmd(cmd, check=True):
    res = subprocess.run(cmd, shell=True, capture_output=True, text=True, cwd="/content")
    if check and res.returncode != 0:
        print(f"{RED}Error: {res.stderr.strip()}{RESET}")
    return res.returncode, res.stdout, res.stderr

if not Heroku_Email or not Heroku_API_Key or not Heroku_App_Name or not Git_Repo_URL:
    print(f"{RED}[!] Heroku Email, API Key, App Name aur Repo URL sabhi bharein!{RESET}")
    sys.exit(1)

if not BOT_TOKEN or not API_ID or not API_HASH or not OWNER_ID or not DATABASE_URL:
    print(f"{RED}[!] Saare 5 variables (BOT_TOKEN, API_ID, API_HASH, OWNER_ID, DATABASE_URL) bharein!{RESET}")
    sys.exit(1)

target_branch = Git_Branch.strip() if Git_Branch.strip() else "master"

env_dict = {
    "BOT_TOKEN": BOT_TOKEN.strip(),
    "API_ID": API_ID.strip(),
    "API_HASH": API_HASH.strip(),
    "OWNER_ID": OWNER_ID.strip(),
    "DATABASE_URL": DATABASE_URL.strip(),
    "WEB_CONCURRENCY": "1"
}

formatted_repo_url = Git_Repo_URL.strip()
if GitHub_Access_Token.strip():
    clean_url = formatted_repo_url.replace("https://", "").replace("http://", "").split("@")[-1]
    formatted_repo_url = f"https://{GitHub_Access_Token.strip()}@{clean_url}"

print(f"{YELLOW}[1/5] Heroku CLI verify ho rahi hai...{RESET}")
subprocess.run("curl -s https://cli-assets.heroku.com/install.sh | sh > /dev/null 2>&1", shell=True, cwd="/content")

print(f"{YELLOW}[2/5] Heroku login config ho raha hai...{RESET}")
netrc_data = f"machine api.heroku.com\n  login {Heroku_Email.strip()}\n  password {Heroku_API_Key.strip()}\nmachine git.heroku.com\n  login {Heroku_Email.strip()}\n  password {Heroku_API_Key.strip()}\n"
with open(os.path.expanduser("~/.netrc"), "w") as f:
    f.write(netrc_data)
os.chmod(os.path.expanduser("~/.netrc"), 0o600)

app_name = Heroku_App_Name.strip().lower()
print(f"{YELLOW}[3/5] App check ki ja rahi hai...{RESET}")
create_code, out, err = run_cmd(f"heroku create {app_name}", check=False)
if create_code != 0:
    info_code, _, _ = run_cmd(f"heroku apps:info -a {app_name}", check=False)
    if info_code != 0:
        print(f"{RED}[!] App Name galat ya already taken hai!{RESET}")
        sys.exit(1)

print(f"{YELLOW}[3.5/5] Buildpacks set ho rahe hain (apt + python)...{RESET}")
run_cmd(f"heroku buildpacks:clear -a {app_name}", check=False)
run_cmd(f"heroku buildpacks:add --index 1 heroku-community/apt -a {app_name}", check=False)
bp_code, _, bp_err = run_cmd(f"heroku buildpacks:add --index 2 heroku/python -a {app_name}", check=False)
if bp_code == 0:
    print(f"{GREEN}✅ Buildpacks set: apt + python{RESET}")
else:
    print(f"{RED}[!] Buildpack set karne mein error: {bp_err.strip()}{RESET}")

print(f"{YELLOW}[*] Variables Heroku par set ho rahe hain...{RESET}")
config_args = [f'{k}="{v}"' for k, v in env_dict.items()]
set_code, _, set_err = run_cmd(f"heroku config:set {' '.join(config_args)} -a {app_name}", check=False)
if set_code == 0:
    print(f"{GREEN}✅ 5 variables set ho gaye!{RESET}")
else:
    print(f"{RED}Config error: {set_err.strip()}{RESET}")

print(f"{YELLOW}[4/5] Repo clone ho raha hai ({target_branch})...{RESET}")
repo_dir = "/content/bot_deploy"
if os.path.exists(repo_dir):
    run_cmd(f"rm -rf {repo_dir}")

code, _, err = run_cmd(f"git clone -b {target_branch} {formatted_repo_url} {repo_dir}")
if code != 0:
    print(f"{RED}[❌] Git Clone fail ho gaya! Branch '{target_branch}' ya token check karein.{RESET}")
    sys.exit(1)

os.chdir(repo_dir)

run_cmd('git config --global user.email "deployer@colab.local"')
run_cmd('git config --global user.name "Colab Deployer"')

# Playwright/Chromium ke liye Heroku build-time hook — agar repo mein
# pehle se nahi hai to yahin bana denge, taaki /ytlogin (agar bot isse
# support karta hai) redeploy/restart ke baad bhi kaam kare.
os.makedirs(os.path.join(repo_dir, "bin"), exist_ok=True)
post_compile_path = os.path.join(repo_dir, "bin", "post_compile")
if not os.path.exists(post_compile_path):
    with open(post_compile_path, "w") as f:
        f.write(
            "#!/usr/bin/env bash\n"
            "set -e\n"
            "echo '-----> Installing Playwright Chromium browser'\n"
            "playwright install chromium 2>/dev/null || true\n"
        )
    os.chmod(post_compile_path, 0o755)
    # IMPORTANT: run_cmd() always runs with cwd="/content" (hardcoded
    # in its definition above), NOT repo_dir — so calling
    # run_cmd("git add ...") here would silently operate on the wrong
    # directory (no .git there) instead of the cloned repo. And even
    # with the right cwd, `git add` alone only STAGES the file — it
    # never reaches the git history that gets pushed below without an
    # explicit commit. Both were real bugs found via testing: the file
    # was created locally but never actually made it into the push.
    # Fixed by running git directly with an explicit cwd=repo_dir for
    # both the add and the commit.
    add_res = subprocess.run(
        "git add bin/post_compile", shell=True, cwd=repo_dir,
        capture_output=True, text=True,
    )
    commit_res = subprocess.run(
        'git commit -m "Add Playwright Chromium build hook (bin/post_compile)"',
        shell=True, cwd=repo_dir, capture_output=True, text=True,
    )
    if commit_res.returncode == 0:
        print(f"{GREEN}✅ bin/post_compile committed{RESET}")
    else:
        print(f"{YELLOW}[!] bin/post_compile commit skipped or failed: {commit_res.stderr.strip()}{RESET}")

print(f"{GREEN}[5/5] Heroku par push ho raha hai...{RESET}\n")
deploy = subprocess.Popen(f"git push https://git.heroku.com/{app_name}.git HEAD:main --force", shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in deploy.stdout:
    print(line, end="")
deploy.wait()

if deploy.returncode == 0:
    print(f"\n{GREEN}✅ Deployed Successfully! Dyno start ho raha hai...{RESET}")
    run_cmd(f"heroku ps:scale worker=1 web=0 -a {app_name}")
    time.sleep(2)
    log_proc = subprocess.Popen(f"heroku logs --tail -a {app_name}", shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    try:
        for log_line in log_proc.stdout:
            print(log_line, end="")
    except KeyboardInterrupt:
        pass
else:
    print(f"\n{RED}❌ Build fail hui.{RESET}")

os.chdir("/content")